[本作品 GitHub](https://github.com/50819just/week07-ui-accessibility-rag)｜[在 Colab 開啟 RAG 01](https://colab.research.google.com/github/50819just/week07-ui-accessibility-rag/blob/main/RAG_01_介面急診室_建立知識庫.ipynb)

# 介面急診室｜RAG 01：建立自己的 WCAG 知識庫

## Goal｜目標與邊界
以 **18 個 WCAG 2.2 成功準則＋4 個詞彙定義**建立向量知識庫。來源為 W3C 2024-12-12 固定版本，原文條件、例外與註解保留；不是完整規範，也不是官方中文翻譯。

本份不使用 LLM 或 Groq Key。主要展示作品在 RAG 02。本作業與碩士論文無關。

## Setup｜執行方式
使用新的 Colab 工作階段，由上往下執行。保留老師固定套件與 EmbeddingGemma 2 → FAISS → 四檔匯出的架構。CPU 可執行；GPU 較快。

## Steps｜建庫流程
公開下載原始文字包 → 擷取正文與 metadata → embedding → FAISS → 完整性檢查 → 匯出 **rag-data.zip**。

**來源辨識：**老師原始 Notebook commit 為 `f424ee3ed3c909830d7df08d17491dacacf88b61`；本副本的資料、下載、人設及介面修改由生成式 AI 協助，未冒稱為使用者獨立撰寫。

## 1. 安裝套件

我們使用 Sentence Transformers 載入模型、NumPy 處理向量，以及 FAISS 搜尋向量。這格固定本教材採用的套件版本，方便記錄與重現。

先執行本格，再執行下格的 import。如果你已在本次執行階段載入其他版本的相關套件，請在安裝後重新啟動工作階段，再從下一格繼續；不需要重跑安裝格。

AI 協助補上 torchvision：純文字模型的 processor 載入仍可能需要此相依；不更換老師固定版本。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1" "torchvision"

## 2. 檢查環境與集中設定

`BATCH_SIZE` 可以調小以降低記憶體使用量。`EXPECTED_DOCUMENT_COUNT` 只用來提醒教材篇數；換自己的資料時可設為 `None`，不會強制只能有16篇。

以下會列出套件版本和 CPU／GPU。Colab 常見的 T4 不支援原生 bfloat16，本程式會改用 float32；不使用 EmbeddingGemma 2 不相容的 float16。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
MODEL_REVISION = "914f7f89142e33e77833254d9c9b90c3cef7303b"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = 22
EXPORT_ZIP_NAME = "rag-data.zip"  # 正式公告指定：這是向量知識庫，不是原始 TXT 包。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

## 3. 下載自己的原始文件包

**AI 協助修改：**將老師手動上傳改成公開直連下載，並檢查 SHA-256；後面的 TXT 安全解壓及建庫架構保留。這個包叫 source-documents.zip，不是 RAG 02 讀取的向量包。

In [ ]:
import io
from urllib.request import Request, urlopen
SOURCE_ZIP_URL = 'https://raw.githubusercontent.com/50819just/week07-ui-accessibility-rag/b5917723e271eee61455b9eba14ad3ef7966487a/source-documents.zip'
EXPECTED_SOURCE_SHA256 = '79391dec9d8ece6db531b385df557c724ce4d04bfc689456f654e10ea9d47fbb'
archive_name = "source-documents.zip"
with urlopen(Request(SOURCE_ZIP_URL, headers={"User-Agent":"Accessibility-RAG-Course/1.0"}), timeout=60) as response:
    archive_bytes = response.read()
SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
if SOURCE_ARCHIVE_SHA256 != EXPECTED_SOURCE_SHA256:
    raise ValueError("原始資料包 SHA-256 不符；停止，避免混用不同版本。")
if not zipfile.is_zipfile(io.BytesIO(archive_bytes)):
    raise ValueError("下載內容不是 ZIP，請檢查公開直連。")
DATA_DIR = Path(tempfile.mkdtemp(prefix="ux_source_"))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [m for m in archive.infolist() if not m.is_dir() and m.filename.lower().endswith(".txt") and m.filename != "W3C_DOCUMENT_LICENSE_2023.txt" and not any(p.startswith((".","__MACOSX")) for p in Path(m.filename).parts)]
    destinations=set()
    for member in members:
        target=(DATA_DIR/member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()) or target in destinations:
            raise ValueError("ZIP 路徑不適當或重複。")
        destinations.add(target); target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(archive.read(member))
txt_paths=sorted([p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower()==".txt"])
if len(txt_paths)!=EXPECTED_DOCUMENT_COUNT:
    raise ValueError(f"預期 {EXPECTED_DOCUMENT_COUNT} 份正文，實際 {len(txt_paths)}。")
print("已下載並驗證",len(txt_paths),"份正文；不匯入 README、授權與測試題。")

## 4. 先看一篇原始文件

每份 TXT 保存標題、W3C 原始小節連結、整理日期、原文發布日期、文件狀態、準則編號、等級與授權。只有【內容】會進入正文；測試題不在資料包中。

In [ ]:
SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

## 5. 第一個可修改函式：取出 RAG 文字

`extract_rag_text(raw_text)` 回傳「標題＋【內容】正文」。找到【內容】後，讀到下一個【區段標題】為止；練習問題和提醒不會混入。

**換自己的資料，主要修改這個函式。** 比如你的文件以 `正文：` 開始，就修改擷取規則。若找不到正文，本函式明確報錯，不擅自把整份文件拿去 embedding。

這個回傳值會原樣保存；未來檢索後也將它交給 LLM，確保搜尋與回答使用同一份內容。

In [ ]:
def extract_rag_text(raw_text):
    """從本教材 TXT 取出標題與正文；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請修改 extract_rag_text() 以配合文件格式。")
    title = title_match.group(1).strip() if title_match else ""
    body = content_match.group("body").strip()
    return f"{title}\n\n{body}" if title else body

sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))

## 6. 取出來源與授權 metadata

保留老師 extract_metadata() 的結構，新增 WCAG 小節、等級、版權與固定版本資訊。metadata 不整包送入 embedding，但回答時一併提供，以便核對。整理日期不等於規範發布日期。

In [ ]:
def extract_metadata(raw_text, filename):
    normalized=raw_text.lstrip("\ufeff").replace("\r\n","\n").replace("\r","\n")
    def read_field(name):
        match=re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$",normalized,flags=re.MULTILINE)
        return (match.group(1).strip() or None) if match else None
    mapping={"title":"標題","original_title":"原文標題","retrieval_alias":"檢索別名","source_url":"來源網址","date":"教材整理日期","source_publication_date":"原文發布日期","source_status":"文件狀態","document_type":"資料類型","sc_id":"準則編號","level":"符合等級","copyright":"版權聲明","license_url":"授權網址","source_snapshot_sha256":"來源快照SHA256"}
    result={k:read_field(v) for k,v in mapping.items()}
    result["title"]=result["title"] or Path(filename).stem
    result["filename"]=filename
    return result
sample_metadata=extract_metadata(sample_raw,sample_path.name)
print(json.dumps(sample_metadata,ensure_ascii=False,indent=2))

## 7. 批次整理成統一格式

把兩個函式組合起來，每篇得到 `doc_id`、`text` 和 `metadata`。檔案先排序，再按順序給編號；只要教材及排序不變，編號便相同。新增文件可能改變編號，因此需要整套重建索引與對照。

請檢查列表中每篇是否有標題、正文及來源。缺少 metadata 會提醒；正文缺失則停止並指出檔名，讓你回到前兩格修改函式。

In [ ]:
documents = []
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    documents.append({"doc_id": f"doc_{number:03d}", "text": rag_text, "metadata": metadata})

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"],
     "正文含標題字元數": len(doc["text"]), "來源": doc["metadata"]["source_url"],
     "整理日期": doc["metadata"]["date"]}
    for doc in documents
])
display(summary)
for doc in documents:
    missing = [key for key in ("source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

## 8. 載入 EmbeddingGemma 2

保留老師純文字模式與精度策略；固定模型 commit，避免日後模型更新造成建庫與查詢不一致。此步不讀取 Groq Key。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

## 9. 檢查輸入長度，避免靜默截斷

本模型採用不同的任務前綴來區分文件和問題。文件本身已有標題，所以組成 `title: none | text: {text}`；問題則用 `task: search result | query: {question}`。

下格計算實際送入模型的文件文字（含前綴）長度。超過上限便停止，不會只取前半段。換成長文件時，可以在後續課程加入 chunking，或重新設計文件單位。

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])





## 10. 產生並觀察文件向量

一篇文件對應矩陣中的一列，欄數是向量維度。`prompt=""` 表示前綴已自行加入，不讓模型額外再加一次。

向量正規化後長度約為1，內積便等於 cosine similarity。下格也將資料轉為 FAISS 使用的 float32 連續陣列，檢查空向量、非有限值及形狀。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

## 11. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 比較問題向量與所有文件向量的內積。因為雙方都正規化，排名就是 cosine similarity 的排名。

FAISS 回傳的是向量編號，**不是原文**。`index_doc_ids` 保存向量順序與文件編號的對應，再透過 `doc_id` 找回正文與 metadata。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

## 12. 定義問題向量化與搜尋函式

問題也要用同一個 embedding 模型向量化，但使用查詢專用前綴。`top_k` 表示取回幾篇文件；程式限制它不超過實際文件數。

搜尋函式只做檢索，還沒有呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. 試查一次，閱讀真正取回的內容

相似度不是正確率。此處只驗證檢索，不會生成回答。可比較 top-k 1、3、5。

In [ ]:
QUESTION = "表單只用紅色框線提示錯誤，沒有文字說明，應該注意什麼？"
TOP_K = 4
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([{"排名":r["rank"],"doc_id":r["doc_id"],"準則":r["metadata"].get("sc_id"),"標題":r["metadata"]["title"],"cosine_similarity":r["score"]} for r in results]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']}")
    print(result["text"])

## 14. 保存索引、文件、編號對照與設定

第二份 Colab 需要下面四個檔案，不能只拿 FAISS 索引：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後提供給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型確切版本、前綴、維度、正規化、套件版本及檢查資訊 |

不保存模型權重、API 金鑰或其他環境變數。第二份 Colab 載入相同模型版本，只重新計算問題向量。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_document_one_vector",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

## 15. 重新載入，驗證第二份 Colab 所需的資料

從剛才保存的檔案重新載入，檢查完整性、文件編號與向量對應，再使用**同一個問題向量**比對保存前後的排名與分數。此處不重算問題向量，以免不同運算裝置的細微浮點誤差干擾索引保存檢查。

也將 FAISS 結果與 NumPy 的全矩陣內積排序比較，確認精確搜尋的意義。遇到完全同分時，兩套工具的同分排列順序可以不同。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

## 16. 打包自己的知識庫

ZIP 根目錄是 index.faiss、documents.json、index_doc_ids.json、config.json。檔名依正式公告為 **rag-data.zip**。模型權重與 API Key 不在裡面；documents.json 含公開原文及授權 metadata。

In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("本機輸出位置：", EXPORT_ZIP)

## Checks｜這份程式確認了什麼？

上一節會實際檢查檔案完整性、ID、維度、保存前後搜尋結果與 NumPy 排序。只有執行後真的通過，才可稱為建庫成功。

## Next Steps｜交給 RAG 02
將本次匯出的 rag-data.zip 放到公開直連位置，記錄 SHA-256；RAG 02 使用相同模型 commit，並透過 AIsuite、Groq 和 Gradio 展示。不要將原始 source-documents.zip 當作索引包。

資料為 W3C WCAG 2.2 節錄，非完整規範。Copyright © 2020-2024 World Wide Web Consortium. [原文固定版本](https://www.w3.org/TR/2024/REC-WCAG22-20241212/)｜[文件授權](https://www.w3.org/copyright/document-license-2023/)。本作品非官方、無 W3C 背書。